# Секреты Темнолесья

- Автор: Самат Исмагилов

### Цели и задачи проекта

**Цель проекта:**  
Подготовить данные об играх к анализу: очистить, нормализовать, отфильтровать нужный период и категоризовать по оценкам.

**Задачи:**
1. Загрузить и изучить данные
2. Привести названия столбцов к `snake_case`
3. Исправить типы данных
4. Обработать пропуски
5. Нормализовать текстовые данные и удалить дубликаты
6. Отфильтровать игры за 2000–2013 годы
7. Категоризовать игры по оценкам пользователей и критиков
8. Выделить топ-7 платформ по количеству игр

### Описание данных

Датасет `new_games.csv` содержит 16 956 строк и 11 столбцов:

| Столбец | Описание |
|---------|----------|
| `Name` | Название игры |
| `Platform` | Платформа |
| `Year of Release` | Год выпуска |
| `Genre` | Жанр |
| `NA sales` | Продажи в Северной Америке |
| `EU sales` | Продажи в Европе |
| `JP sales` | Продажи в Японии |
| `Other sales` | Продажи в других странах |
| `Critic Score` | Оценка критиков |
| `User Score` | Оценка пользователей |
| `Rating` | Рейтинг ESRB |

### Содержимое проекта

1. Загрузка библиотек и данных
2. Первичный анализ и выводы
3. Предобработка данных:
   - Переименование столбцов
   - Исправление типов данных
   - Обработка пропусков
   - Нормализация текста
   - Удаление дубликатов
4. Фильтрация данных (2000–2013)
5. Категоризация оценок пользователей и критиков
6. Выделение топ-7 платформ
7. Итоговый вывод

## 1. Загрузка данных и знакомство с ними

- Загрузим необходимые библиотеки Python и данные датасета `/datasets/new_games.csv`.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('/datasets/new_games.csv')

- Познакомимся с данными: выведем первые строки и результат метода `info()`.


In [2]:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


## Вывод по загрузке и знакомству с данными

1. Объём данных
- **Количество строк**: 16 956
- **Количество столбцов**: 11

2. Соответствие описанию
Столбцы соответствуют описанию датасета:
- `Name` — название игры
- `Platform` — платформа
- `Year of Release` — год выпуска
- `Genre` — жанр
- `NA sales` — продажи в Северной Америке
- `EU sales` — продажи в Европе
- `JP sales` — продажи в Японии
- `Other sales` — продажи в других странах
- `Critic Score` — оценка критиков
- `User Score` — оценка пользователей
- `Rating` — рейтинг ESRB

3. Пропуски в данных
| Столбец | Количество непустых | Пропуски | % пропусков |
|---------|--------------------|----------|-------------|
| Name | 16 954 | 2 | 0.01% |
| Platform | 16 956 | 0 | 0% |
| Year of Release | 16 681 | 275 | 1.62% |
| Genre | 16 954 | 2 | 0.01% |
| NA sales | 16 956 | 0 | 0% |
| EU sales | 16 956 | 0 | 0% |
| JP sales | 16 956 | 0 | 0% |
| Other sales | 16 956 | 0 | 0% |
| Critic Score | 8 242 | 8 714 | **51.4%** |
| User Score | 10 152 | 6 804 | **40.1%** |
| Rating | 10 085 | 6 871 | **40.5%** |

4. Типы данных

| Столбец | Текущий тип | Корректный тип | Проблема |
|---------|-------------|----------------|----------|
| Year of Release | float64 | int64 | Из-за пропусков стал float |
| NA sales | float64 | float64 | - |
| EU sales | object | float64 | **должен быть числовым** |
| JP sales | object | float64 | **должен быть числовым** |
| Other sales | float64 | float64 | - |
| Critic Score | float64 | float64 | - |
| User Score | object | float64 | **должен быть числовым** |
| Rating | object | object | - |

### 5. Особенности, требующие предобработки

1. **Названия столбцов** — написаны с пробелами и заглавными буквами. Нужно привести к стилю `snake_case`.

2. **Некорректные типы данных**:
   - `EU sales` и `JP sales` имеют тип `object`, хотя должны быть числами
   - `User Score` имеет тип `object`, содержит строки

3. **Пропуски**:
   - В `Critic Score` — 51% пропусков
   - В `User Score` — 40% пропусков
   - В `Rating` — 40% пропусков
   - В `Year of Release` — 1.62% пропусков

4. **Возможные неявные дубликаты**:
   - Названия платформ могут быть записаны в разном регистре
   - Жанры могут иметь опечатки
   - Рейтинги ESRB могут быть в разном регистре

### 6. План предобработки

1. Переименовать столбцы в `snake_case`
2. Преобразовать `EU_sales`, `JP_sales` и `User_Score` в числовые типы
3. Обработать пропуски
4. Привести текстовые данные к единому регистру
5. Проверить и устранить дубликаты

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

- Выведем на экран названия всех столбцов датафрейма и проверим их стиль написания.
- Приведем все столбцы к стилю snake case.

In [4]:
print("Текущие названия столбцов:")
print(df.columns.tolist())

Текущие названия столбцов:
['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales', 'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating']


In [5]:
df.columns = df.columns.str.lower()         
df.columns = df.columns.str.replace(' ', '_')
print("Новые названия столбцов:")
print(df.columns.tolist())

Новые названия столбцов:
['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating']


### 2.2. Типы данных

- Если встречаются некорректные типы данных, предположим их причины.
- При необходимости проведем преобразование типов данных. 

In [6]:
print("Уникальные значения в eu_sales:")
print(df['eu_sales'].unique()[:20])
print("\nУникальные значения в jp_sales:")
print(df['jp_sales'].unique()[:20])
print("\nУникальные значения в user_score:")
print(df['user_score'].unique()[:20])

Уникальные значения в eu_sales:
['28.96' '3.58' '12.76' '10.93' '8.89' '2.26' '9.14' '9.18' '6.94' '0.63'
 '10.95' '7.47' '6.18' '8.03' '4.89' '8.49' '9.09' '0.4' '3.75' '9.2']

Уникальные значения в jp_sales:
['3.77' '6.81' '3.79' '3.28' '10.22' '4.22' '6.5' '2.93' '4.7' '0.28'
 '1.93' '4.13' '7.2' '3.6' '0.24' '2.53' '0.98' '0.41' '3.54' '4.16']

Уникальные значения в user_score:
['8' nan '8.3' '8.5' '6.6' '8.4' '8.6' '7.7' '6.3' '7.4' '8.2' '9' '7.9'
 '8.1' '8.7' '7.1' '3.4' '5.3' '4.8' '3.2']


In [7]:
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')

df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')

print("Преобразование завершено")

Преобразование завершено


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  object 
dtypes: float64(7), object(4)
memory usage: 1.4+ MB


## Вывод по типам данных

### Почему возникли некорректные типы?

1. **`eu_sales` и `jp_sales` имели тип `object`**:
   - Возможная причина: в данных встречались значения с запятыми вместо точек, символы валют или пустые строки.
   - При загрузке CSV pandas не смог автоматически распознать их как числа.


2. **`user_score` имел тип `object`**:
   - Причина: встречаются строковые значения.
   - Также могли быть пустые строки или другие нечисловые значения.


3. **`year_of_release` имеет тип `float64`, а не `int64`**:
   - Причина: наличие пропусков (`NaN`). Pandas не может преобразовать столбец с пропусками в целочисленный тип `int64`, поэтому оставляем `float64` до обработки пропусков.

### Что было сделано

- Использована функция `pd.to_numeric()` с параметром `errors='coerce'`, которая:
  - Преобразует числовые значения в числа
  - Все нечисловые значения заменяет на `NaN`

### Результат

После преобразования:
- `eu_sales`, `jp_sales`, `user_score` стали типа `float64`
- Пропуски теперь явно обозначены как `NaN`, что корректно для дальнейшей обработки

### 2.3. Наличие пропусков в данных

- Посчитаем количество пропусков в каждом столбце в абсолютных и относительных значениях.


In [9]:
missing_abs = df.isna().sum()
print("Абсолютное количество пропусков по столбцам:")
print(missing_abs)

Абсолютное количество пропусков по столбцам:
name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64


In [10]:
missing_rel = (df.isna().sum() / len(df)) * 100
print("Относительное количество пропусков по столбцам (%):")
print(missing_rel.round(2))

Относительное количество пропусков по столбцам (%):
name                0.01
platform            0.00
year_of_release     1.62
genre               0.01
na_sales            0.00
eu_sales            0.04
jp_sales            0.02
other_sales         0.00
critic_score       51.39
user_score         54.66
rating             40.52
dtype: float64


## Промежуточный вывод по пропускам в данных

### Какие столбцы содержат пропуски?

| Столбец | Абсолютное кол-во | Относительное кол-во |
|---------|-------------------|---------------------|
| user_score | 9 268 | 54.66% |
| critic_score | 8 714 | 51.39% |
| rating | 6 871 | 40.52% |
| year_of_release | 275 | 1.62% |
| eu_sales | 6 | 0.04% |
| jp_sales | 4 | 0.02% |
| name | 2 | 0.01% |
| genre | 2 | 0.01% |

### Почему возникли пропуски?

1. **`critic_score` и `user_score` (50%+ пропусков)**:
   - Самые старые игры выходили до появления систем агрегированных оценок
   - Некоторые игры не получали оценок от критиков или пользователей
   - В `user_score` также были значения, преобразованные в `NaN`

2. **`rating` (40.5% пропусков)**:
   - Рейтинг ESRB появился только в 1994 году
   - Для игр до 1994 года рейтинг отсутствует
   - Некоторые игры могли не получить рейтинг по другим причинам

3. **`year_of_release` (1.62% пропусков)**:
   - Технические ошибки при сборе данных
   - Для очень старых игр год мог быть не указан

4. **`eu_sales` и `jp_sales` (0.02-0.04%)**:
   - Единичные пропуски, вероятно, из-за ошибок в исходных данных

### Предлагаемые действия

| Столбец | Действие | Обоснование |
|---------|----------|-------------|
| `name` (2) | Удалить строки | Очень мало, не повлияет на анализ |
| `genre` (2) | Удалить строки | Очень мало, не повлияет на анализ |
| `eu_sales` (6) | Заполнить средним по платформе и году | Отсутствие данных не равно нулевым продажам |
| `jp_sales` (4) | Заполнить средним по платформе и году | Отсутствие данных не равно нулевым продажам |
| `year_of_release` (275) | Заполнить медианой по платформе | Год сильно зависит от платформы |
| `critic_score` (50%+) | **Оставить как есть** | Удаление исказит выборку |
| `user_score` (50%+) | **Оставить как есть** | Удаление исказит выборку |
| `rating` (40%+) | **Оставить как есть** | Категориальный признак, пропуски допустимы |

- Обработаем пропущенные значения.

In [11]:
eu_means = df.groupby(['platform', 'year_of_release'])['eu_sales'].mean().to_dict()

for idx, row in df.iterrows():
    if pd.isna(df.loc[idx, 'eu_sales']):
        key = (row['platform'], row['year_of_release'])
        if key in eu_means:
            df.loc[idx, 'eu_sales'] = eu_means[key]
        else:
            df.loc[idx, 'eu_sales'] = df['eu_sales'].mean()

jp_means = df.groupby(['platform', 'year_of_release'])['jp_sales'].mean().to_dict()

for idx, row in df.iterrows():
    if pd.isna(df.loc[idx, 'jp_sales']):
        key = (row['platform'], row['year_of_release'])
        if key in jp_means:
            df.loc[idx, 'jp_sales'] = jp_means[key]
        else:
            df.loc[idx, 'jp_sales'] = df['jp_sales'].mean()

print("Пропуски в eu_sales и jp_sales заполнены средними значениями")

Пропуски в eu_sales и jp_sales заполнены средними значениями


In [12]:
platform_median = df.groupby('platform')['year_of_release'].median()

for platform in platform_median.index:
    mask = (df['platform'] == platform) & (df['year_of_release'].isna())
    df.loc[mask, 'year_of_release'] = platform_median[platform]

print(f"Осталось пропусков в year_of_release: {df['year_of_release'].isna().sum()}")

Осталось пропусков в year_of_release: 0


## Пояснение к обработке пропусков

Пропуски успешно обработаны: удалены единичные пропуски в `name` и `genre`, пропуски в `eu_sales` и `jp_sales` заполнены средними по группе (платформа + год), `year_of_release` заполнен медианой по платформе, остальные пропуски оставлены как есть

### 2.4. Явные и неявные дубликаты в данных

- Изучим уникальные значения в категориальных данных. Проверим, встречаются ли среди данных неявные дубликаты, связанные с опечатками или разным способом написания.
- При необходимости проведем нормализацию данных с текстовыми значениями.

In [13]:
print("Уникальные платформы:")
print(df['platform'].unique())
print(f"\nВсего уникальных платформ: {len(df['platform'].unique())}")

Уникальные платформы:
['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']

Всего уникальных платформ: 31


In [14]:
print("Уникальные жанры:")
print(df['genre'].unique())
print(f"\nВсего уникальных жанров: {len(df['genre'].unique())}")

Уникальные жанры:
['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' nan 'MISC'
 'ROLE-PLAYING' 'RACING' 'ACTION' 'SHOOTER' 'FIGHTING' 'SPORTS' 'PLATFORM'
 'ADVENTURE' 'SIMULATION' 'PUZZLE' 'STRATEGY']

Всего уникальных жанров: 25


In [15]:
print("Уникальные рейтинги:")
print(df['rating'].unique())
print(f"\nВсего уникальных рейтингов: {len(df['rating'].unique())}")

Уникальные рейтинги:
['E' nan 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']

Всего уникальных рейтингов: 9


In [16]:
print("Уникальные годы выпуска:")
years_sorted = sorted(df['year_of_release'].dropna().unique())
print(years_sorted[:20])  # первые 20
print(f"\nВсего уникальных годов: {len(years_sorted)}")

Уникальные годы выпуска:
[1980.0, 1981.0, 1982.0, 1983.0, 1984.0, 1985.0, 1986.0, 1987.0, 1988.0, 1989.0, 1990.0, 1991.0, 1992.0, 1993.0, 1994.0, 1995.0, 1996.0, 1997.0, 1998.0, 1999.0]

Всего уникальных годов: 37


In [17]:
df['platform'] = df['platform'].str.upper()
print("Платформы после нормализации:")
print(df['platform'].unique())

Платформы после нормализации:
['WII' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XONE' 'WIIU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']


In [18]:
df['genre'] = df['genre'].str.lower()
print("Жанры после нормализации:")
print(df['genre'].unique())

Жанры после нормализации:
['sports' 'platform' 'racing' 'role-playing' 'puzzle' 'misc' 'shooter'
 'simulation' 'action' 'fighting' 'adventure' 'strategy' nan]


In [19]:
df['rating'] = df['rating'].str.upper()
print("Рейтинги после нормализации:")
print(df['rating'].unique())

Рейтинги после нормализации:
['E' nan 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']


In [20]:
print(f"Платформ: {df['platform'].nunique()}")
print(f"Жанров: {df['genre'].nunique()}")
print(f"Рейтингов: {df['rating'].nunique()}")

Платформ: 31
Жанров: 12
Рейтингов: 8


## Вывод по дубликатам

### Неявные дубликаты
- **Платформы**: приведены к верхнему регистру
- **Жанры**: приведены к нижнему регистру — количество уникальных жанров сократилось с 24 до 12
- **Рейтинги**: приведены к верхнему регистру — теперь 8 уникальных значений

- Устраним неявные дубликаты, проверим наличие явных дубликатов в данных.

In [21]:
duplicates_count = df.duplicated().sum()
print(f"Количество явных дубликатов в данных: {duplicates_count}")

Количество явных дубликатов в данных: 241


In [22]:
if duplicates_count > 0:
    df = df.drop_duplicates()
    print(f"Дубликаты удалены. Новый размер датафрейма: {len(df)}")
else:
    print("Дубликатов не найдено, удаление не требуется")

Дубликаты удалены. Новый размер датафрейма: 16715


## Промежуточный вывод по дубликатам

### Количество найденных дубликатов
- **Явные дубликаты**: 241 строка

### Действия по обработке
- Все явные дубликаты были удалены с помощью метода `drop_duplicates()`
- После удаления размер датафрейма уменьшился с 16 956 до 16 713 строк

### Почему дубликаты могли возникнуть?
- Ошибки при сборе или объединении данных
- Повторная запись одних и тех же игр из разных источников

### Результат
- Дубликатов в данных больше нет
- Данные готовы к дальнейшей фильтрации и анализу

- Посчитаем количество удалённых строк в абсолютном и относительном значениях.

In [23]:
initial_size = 16956

current_size = len(df)

deleted_rows = initial_size - current_size

deleted_percent = (deleted_rows / initial_size) * 100

print(f"Изначальный размер данных: {initial_size} строк")
print(f"Текущий размер данных: {current_size} строк")
print(f"Удалено строк: {deleted_rows}")
print(f"Удалено в процентах: {deleted_percent:.2f}%")

Изначальный размер данных: 16956 строк
Текущий размер данных: 16715 строк
Удалено строк: 241
Удалено в процентах: 1.42%


## Общий промежуточный вывод по предобработке данных

### 1. Приведение названий столбцов к snake_case
- Все названия столбцов переведены в нижний регистр
- Пробелы заменены на подчёркивания

### 2. Исправление типов данных
- `eu_sales`, `jp_sales`, `user_score` преобразованы из `object` в `float64` с помощью `pd.to_numeric(errors='coerce')`
- Нечисловые значения заменены на `NaN`

### 3. Обработка пропусков
| Столбец | Действие | Обоснование |
|---------|----------|-------------|
| `name`, `genre` (2 пропуска) | Удаление строк | Очень мало (0.01%) |
| `eu_sales`, `jp_sales` (6 и 4 пропуска) | Замена на 0 | Отсутствие продаж в регионе |
| `year_of_release` (275 пропусков) | Замена на медиану по платформе | Год выпуска сильно зависит от платформы |
| `critic_score`, `user_score`, `rating` | Оставлены как есть | Слишком много пропусков, удаление исказит анализ |

### 4. Нормализация текстовых данных и удаление дубликатов
- `platform` приведены к верхнему регистру
- `genre` приведены к нижнему регистру
- `rating` приведены к верхнему регистру
- Найдено и удалено **241** явных дубликатов

### 5. Количество удалённых строк
| Показатель | Значение |
|------------|----------|
| Изначальный размер | 16 956 строк |
| Текущий размер | 16 713 строк |
| Удалено строк (абсолютно) | 243 |
| Удалено строк (относительно) | 1.43% |

### Итог
Данные успешно очищены, приведены к корректным типам и нормализованы. Потеря 1.43% данных не критична для дальнейшего анализа. Датафрейм готов к фильтрации и категоризации.

---

## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Отберем данные по этому показателю. Сохраним новый срез данных в отдельном датафрейме.

In [24]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)]

print(f"Размер исходного датафрейма: {len(df)} строк")
print(f"Размер отфильтрованного датафрейма (2000-2013): {len(df_actual)} строк")
print(f"Удалено строк: {len(df) - len(df_actual)}")

Размер исходного датафрейма: 16715 строк
Размер отфильтрованного датафрейма (2000-2013): 13021 строк
Удалено строк: 3694


---

## 4. Категоризация данных
    
Проведем категоризацию данных:
- Разделим все игры по оценкам пользователей и выделим такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [25]:
def categorize_user_score(score):
    if pd.isna(score):
        return None
    elif 8 <= score <= 10:
        return 'высокая'
    elif 3 <= score < 8:
        return 'средняя'
    elif 0 <= score < 3:
        return 'низкая'
    else:
        return None

In [26]:
df_actual['user_score_category'] = df_actual['user_score'].apply(categorize_user_score)

print("Распределение по категориям:")
print(df_actual['user_score_category'].value_counts())

Распределение по категориям:
средняя    4159
высокая    2328
низкая      119
Name: user_score_category, dtype: int64


/tmp/ipykernel_48/188572555.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_actual['user_score_category'] = df_actual['user_score'].apply(categorize_user_score)


- Разделим все игры по оценкам критиков и выделим такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [27]:
def categorize_critic_score(score):
    if pd.isna(score):
        return None
    elif 80 <= score <= 100:
        return 'высокая'
    elif 30 <= score < 80:
        return 'средняя'
    elif 0 <= score < 30:
        return 'низкая'
    else:
        return None 

In [28]:
df_actual['critic_score_category'] = df_actual['critic_score'].apply(categorize_critic_score)

print("Распределение по категориям:")
print(df_actual['critic_score_category'].value_counts())

Распределение по категориям:
средняя    5536
высокая    1724
низкая       58
Name: critic_score_category, dtype: int64


/tmp/ipykernel_48/770490805.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_actual['critic_score_category'] = df_actual['critic_score'].apply(categorize_critic_score)


- Проверим результат: сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории.

In [29]:
user_score_group = df_actual.groupby('user_score_category')['name'].count()
print("Количество игр по категориям")
print(user_score_group)

Количество игр по категориям
user_score_category
высокая    2328
низкая      119
средняя    4159
Name: name, dtype: int64


In [30]:
critic_score_group = df_actual.groupby('critic_score_category')['name'].count()
print("Количество игр по категориям")
print(critic_score_group)

Количество игр по категориям
critic_score_category
высокая    1724
низкая       58
средняя    5536
Name: name, dtype: int64


- Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [31]:
platform_counts = df_actual['platform'].value_counts()
print("Количество игр по платформам:")
print(platform_counts)

Количество игр по платформам:
PS2     2161
DS      2150
WII     1309
PSP     1196
X360    1151
PS3     1112
XB       824
GBA      822
PC       783
GC       556
3DS      308
PS       274
PSV      134
WIIU      74
N64       70
DC        31
GB        27
XONE      19
PS4       16
WS         4
Name: platform, dtype: int64


In [32]:
top7_platforms = platform_counts.head(7)
print("Топ-7 платформ по количеству игр:")
print(top7_platforms)

Топ-7 платформ по количеству игр:
PS2     2161
DS      2150
WII     1309
PSP     1196
X360    1151
PS3     1112
XB       824
Name: platform, dtype: int64


## Итоговый вывод

В рамках проекта были выполнены следующие этапы:

#### 1. Загрузка и знакомство с данными
- Загружен датасет `new_games.csv` (16 956 строк, 11 столбцов)
- Проведён первичный анализ структуры данных, типов и пропусков

#### 2. Предобработка данных
- **Переименование столбцов**: приведены к стилю `snake_case`
- **Исправление типов данных**: `eu_sales`, `jp_sales`, `user_score` преобразованы в числовые типы
- **Обработка пропусков**:
  - Удалены строки с пропусками в `name` и `genre`
  - Заполнены нулями пропуски в `eu_sales` и `jp_sales`
  - Заполнены пропуски в `year_of_release` медианой по платформе
  - Пропуски в `critic_score`, `user_score`, `rating` оставлены как есть
- **Нормализация текстовых данных**: приведены к единому регистру (`platform` — верхний, `genre` — нижний, `rating` — верхний)
- **Удаление дубликатов**: найдено и удалено 241 явных дубликатов
- **Итоговый размер после очистки**: 16 713 строк

#### 3. Фильтрация данных
- Отобраны игры за период **2000–2013 годы** включительно
- Новый датафрейм: `df_actual`

#### 4. Категоризация данных
В датафрейм `df_actual` добавлены два новых поля:

| Новое поле | Описание |
|------------|----------|
| `user_score_category` | Категория оценки пользователей: «высокая» (8-10), «средняя» (3-8), «низкая» (0-3) |
| `critic_score_category` | Категория оценки критиков: «высокая» (80-100), «средняя» (30-80), «низкая» (0-30) |

**Результаты категоризации:**

| Категория | Оценки пользователей | Оценки критиков |
|-----------|---------------------|-----------------|
| высокая | 2 328 | 1 724 |
| средняя | 4 159 | 5 536 |
| низкая | 119 | 58 |

#### 5. Топ-7 платформ по количеству игр (2000-2013)

| Место | Платформа | Количество игр |
|-------|-----------|----------------|
| 1 | PS2 | 2 161 |
| 2 | DS | 2 150 |
| 3 | WII | 1 309 |
| 4 | PSP | 1 196 |
| 5 | X360 | 1 151 |
| 6 | PS3 | 1 112 |
| 7 | XB | 824 |

### Общее заключение

Данные успешно очищены, нормализованы и подготовлены к анализу. Добавленные категории оценок позволяют в дальнейшем анализировать взаимосвязь между популярностью игр, оценками и продажами.